In [ ]:
import os
import pandas as pd
from openai import OpenAI

# 1. Inicialize o cliente da OpenAI (certifique-se de configurar a variável de ambiente OPENAI_API_KEY)
client = OpenAI(api_key="YOUR_API_KEY_HERE")
def classificar_avaliacao(texto_avaliacao):
    """Envia o texto da avaliação para a API e retorna a categoria."""
    if not isinstance(texto_avaliacao, str) or str(texto_avaliacao).strip() == "":
        return "Invalido"
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", # Modelo rápido e barato para classificação
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Você é um classificador de dados especialista. Seu trabalho é ler avaliações de clientes "
                        "e classificá-lo estritamente em uma das seguintes categorias: [Positiva, Neutra, Negativa]. "
                        "Responda apenas com a palavra da categoria, sem justificativas, pontuações ou texto adicional."
                    )
                },
                {
                    "role": "user",
                    "content": f"Classifique esta avaliação: '{texto_avaliacao}'"
                }
            ],
            temperature=0, # Garante respostas mais determinísticas e consistentes
            max_tokens=10
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Erro ao processar: {e}")
        return "Erro"

df = pd.read_csv('../../Dados/olist_order_reviews_dataset.csv')

# 3. Aplique a função de classificação em lote no seu DataFrame
print("Iniciando a classificação...")

df['avaliacao'] = df['review_comment_message'].combine_first(df['review_comment_title'])
df['avaliacao'] = df['avaliacao'].apply(classificar_avaliacao)

df.to_csv('dataset_classificado.csv', index=False, encoding='utf-8')
print("Classificação concluída! Arquivo salvo como 'dataset_classificado.csv'.")
print(df)
